# YOLO → COCO Annotation Conversion

Converts annotations from YOLO format (.txt per image) to COCO format (single .json file).

**YOLO format** (each line in a .txt):
```
class_id  x_center  y_center  width  height
(normalised values between 0 and 1)
```

**COCO format** (JSON structure):
```json
{
    "images":      [ { id, file_name, width, height }, ... ],
    "annotations": [ { id, image_id, category_id, bbox, area }, ... ],
    "categories":  [ { id, name }, ... ]
}
```

In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm

## Configuration

In [ ]:
CLASS_NAMES = [
    "Button", "Checkbox", "Dropdown", "Icon", "Image", "Input",
    "Link", "Menu", "Modal", "Progress_bar", "Radio_button",
    "Slider", "Tab", "Text", "Text_field", "Toolbar",
]

SUBSETS = {
    "train": {
        "images":      "/content/drive/MyDrive/dataset_split/train/images",
        "labels":      "/content/drive/MyDrive/dataset_split/train/labels",
        "output_json": "/content/drive/MyDrive/dataset_split/train/annotations/train.json",
    },
    "val": {
        "images":      "/content/drive/MyDrive/dataset_split/val/images",
        "labels":      "/content/drive/MyDrive/dataset_split/val/labels",
        "output_json": "/content/drive/MyDrive/dataset_split/val/annotations/val.json",
    },
    "test": {
        "images":      "/content/drive/MyDrive/dataset_split/test/images",
        "labels":      "/content/drive/MyDrive/dataset_split/test/labels",
        "output_json": "/content/drive/MyDrive/dataset_split/test/annotations/test.json",
    },
}

## Conversion Functions

In [ ]:
def create_categories(class_names):
    """Creates the COCO categories list from class names."""
    return [{"id": i + 1, "name": name} for i, name in enumerate(class_names)]


def convert_yolo_to_coco(images_dir, labels_dir, class_names):
    """
    Reads all YOLO .txt files and converts to COCO structure.

    Args:
        images_dir:  path to the images folder
        labels_dir:  path to the YOLO .txt labels folder
        class_names: ordered list of class names

    Returns:
        Dictionary in COCO format with images, annotations, and categories.
    """
    coco_images = []
    coco_annotations = []
    coco_categories = create_categories(class_names)

    annotation_id = 0

    valid_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    image_files = sorted([
        f for f in os.listdir(images_dir)
        if f.lower().endswith(valid_extensions)
    ])

    print(f"Found {len(image_files)} images in {images_dir}")

    for image_id, filename in enumerate(tqdm(image_files, desc="  Converting")):
        img_path = os.path.join(images_dir, filename)
        with Image.open(img_path) as img:
            img_width, img_height = img.size

        coco_images.append({
            "id":        image_id,
            "file_name": filename,
            "width":     img_width,
            "height":    img_height,
        })

        base_name = os.path.splitext(filename)[0]
        label_path = os.path.join(labels_dir, base_name + ".txt")

        if not os.path.exists(label_path):
            continue

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            w_norm   = float(parts[3])
            h_norm   = float(parts[4])

            w_px  = w_norm * img_width
            h_px  = h_norm * img_height
            x_min = (x_center * img_width) - (w_px / 2)
            y_min = (y_center * img_height) - (h_px / 2)

            x_max = x_min + w_px
            y_max = y_min + h_px
            x_min = max(0.0, x_min)
            y_min = max(0.0, y_min)
            x_max = min(x_max, float(img_width))
            y_max = min(y_max, float(img_height))
            w_px  = x_max - x_min
            h_px  = y_max - y_min

            area = w_px * h_px

            coco_annotations.append({
                "id":          annotation_id,
                "image_id":    image_id,
                "category_id": class_id + 1,
                "bbox":        [round(x_min, 2), round(y_min, 2),
                                round(w_px, 2), round(h_px, 2)],
                "area":        round(area, 2),
                "segmentation": [],
                "iscrowd":     0,
            })
            annotation_id += 1

    return {
        "images":      coco_images,
        "annotations": coco_annotations,
        "categories":  coco_categories,
    }


def save_json(data, output_path):
    """Saves the COCO dictionary as a JSON file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"JSON saved to: {output_path}")

## Run Conversion

In [ ]:
print("YOLO → COCO Annotation Conversion")

for subset_name, paths in SUBSETS.items():
    images_dir  = paths["images"]
    labels_dir  = paths["labels"]
    output_json = paths["output_json"]

    print(f"\n Processing subset: {subset_name}")

    if not os.path.isdir(images_dir):
        print(f"Images folder not found: {images_dir} — skipping.")
        continue
    if not os.path.isdir(labels_dir):
        print(f"Labels folder not found: {labels_dir} — skipping.")
        continue

    coco_data = convert_yolo_to_coco(images_dir, labels_dir, CLASS_NAMES)

    n_imgs  = len(coco_data["images"])
    n_anots = len(coco_data["annotations"])
    print(f"Result: {n_imgs} images, {n_anots} annotations")

    save_json(coco_data, output_json)

print("\n Conversion complete!")